In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import ast
from scipy import stats
from sklearn.metrics import auc as compute_auc
import warnings
warnings.filterwarnings('ignore')

# Configurazione stile per pubblicazione scientifica
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 13

In [2]:
# ============================================================================
# SEZIONE 1: CARICAMENTO E PREPARAZIONE DATI
# ============================================================================

def load_experiment_data(base_path):
    """
    Carica tutti i dati degli esperimenti
    
    Args:
        base_path: Path alla directory principale degli esperimenti
    
    Returns:
        dict: Dizionario con history e metrics per ogni esperimento
    """
    base_path = Path(base_path)
    experiments = {
        'IAM→IAM': 'mobilenet_v3_small_iam_to_iam',
        'IAM→RIMES': 'mobilenet_v3_small_iam_to_rimes',
        'RIMES→IAM': 'mobilenet_v3_small_rimes_to_iam',
        'RIMES→RIMES': 'mobilenet_v3_small_rimes_to_rimes'
    }
    
    data = {}
    for exp_name, exp_dir in experiments.items():
        exp_path = base_path / exp_dir
        
        # Carica history
        history_file = exp_path / f"{exp_dir}_history.csv"
        history = pd.read_csv(history_file)
        
        # Carica metrics
        metrics_file = exp_path / f"{exp_dir}_final_metrics.csv"
        metrics = pd.read_csv(metrics_file)
        
        data[exp_name] = {
            'history': history,
            'metrics': metrics,
            'path': exp_path
        }
    
    return data

def parse_distance_array(dist_string):
    """Converte stringa di array numpy in array vero"""
    # Rimuovi spazi extra e converti
    clean_string = dist_string.strip()
    if clean_string.startswith('[') and clean_string.endswith(']'):
        return np.fromstring(clean_string[1:-1], sep=' ')
    return ast.literal_eval(dist_string)

In [3]:
# ============================================================================
# SEZIONE 2: ANALISI DELLE CURVE DI TRAINING
# ============================================================================

def plot_training_curves(data, save_path=None):
    """
    Genera grafici delle curve di training e validation loss
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    
    for idx, (exp_name, exp_data) in enumerate(data.items()):
        ax = axes[idx]
        history = exp_data['history']
        
        # Plot losses
        epochs = history['epoch']
        ax.plot(epochs, history['train_loss'], label='Training Loss', 
                linewidth=2, marker='o', markersize=3, alpha=0.8)
        ax.plot(epochs, history['val_loss'], label='Validation Loss', 
                linewidth=2, marker='s', markersize=3, alpha=0.8)
        
        # Trova minimo validation loss
        min_val_idx = history['val_loss'].idxmin()
        min_val_loss = history.loc[min_val_idx, 'val_loss']
        min_val_epoch = history.loc[min_val_idx, 'epoch']
        
        ax.axvline(min_val_epoch, color='red', linestyle='--', 
                   alpha=0.5, linewidth=1.5, label=f'Best epoch ({min_val_epoch})')
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Binary Cross-Entropy Loss')
        ax.set_title(f'{exp_name}\n(Min Val Loss: {min_val_loss:.4f})')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

def analyze_convergence(data):
    """
    Analizza la convergenza del training
    """
    convergence_stats = []
    
    for exp_name, exp_data in data.items():
        history = exp_data['history']
        
        # Statistiche di convergenza
        final_train_loss = history['train_loss'].iloc[-1]
        final_val_loss = history['val_loss'].iloc[-1]
        min_val_loss = history['val_loss'].min()
        best_epoch = history['val_loss'].idxmin() + 1
        total_epochs = len(history)
        
        # Overfitting indicator
        overfitting = final_val_loss - min_val_loss
        
        # Generalization gap
        gen_gap = final_val_loss - final_train_loss
        
        convergence_stats.append({
            'Experiment': exp_name,
            'Best Epoch': best_epoch,
            'Total Epochs': total_epochs,
            'Final Train Loss': final_train_loss,
            'Final Val Loss': final_val_loss,
            'Best Val Loss': min_val_loss,
            'Overfitting Δ': overfitting,
            'Generalization Gap': gen_gap
        })
    
    df_conv = pd.DataFrame(convergence_stats)
    return df_conv

In [4]:
# ============================================================================
# SEZIONE 3: ANALISI DELLE PERFORMANCE BIOMETRICHE
# ============================================================================

def create_performance_summary(data):
    """
    Crea tabella riassuntiva delle performance
    """
    summary = []
    
    for exp_name, exp_data in data.items():
        metrics = exp_data['metrics'].iloc[0]
        
        summary.append({
            'Experiment': exp_name,
            'AUC': metrics['auc'],
            'EER (%)': metrics['eer'] * 100,
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1-Score': metrics['f1'],
            "d'": metrics['d_prime'],
            'Decidability': metrics['decidability'],
            'ACC@FAR=0.1%': metrics['acc_far_0.001'],
            'ACC@FAR=1%': metrics['acc_far_0.01']
        })
    
    df_summary = pd.DataFrame(summary)
    return df_summary

def plot_roc_curves(data, save_path=None):
    """
    Plotta le curve ROC per tutti gli esperimenti
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.Set2(np.linspace(0, 1, 4))
    
    for idx, (exp_name, exp_data) in enumerate(data.items()):
        metrics = exp_data['metrics'].iloc[0]
        
        # Parse FPR e TPR
        fpr = parse_distance_array(metrics['fpr'])
        tpr = parse_distance_array(metrics['tpr'])
        auc_score = metrics['auc']
        
        ax.plot(fpr, tpr, linewidth=2.5, label=f'{exp_name} (AUC={auc_score:.4f})',
                color=colors[idx], alpha=0.9)
    
    # Linea diagonale (random classifier)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5, label='Random Classifier')
    
    ax.set_xlabel('False Positive Rate (FAR)')
    ax.set_ylabel('True Positive Rate (TAR)')
    ax.set_title('Receiver Operating Characteristic (ROC) Curves\nCross-Dataset Writer Verification')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

def plot_det_curves(data, save_path=None):
    """
    Plotta le curve DET (Detection Error Tradeoff)
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.Set2(np.linspace(0, 1, 4))
    
    for idx, (exp_name, exp_data) in enumerate(data.items()):
        metrics = exp_data['metrics'].iloc[0]
        
        fpr = parse_distance_array(metrics['fpr'])
        tpr = parse_distance_array(metrics['tpr'])
        fnr = 1 - tpr  # False Negative Rate
        
        # DET curve usa scala log-log
        ax.plot(fpr * 100, fnr * 100, linewidth=2.5, 
                label=f'{exp_name} (EER={metrics["eer"]*100:.2f}%)',
                color=colors[idx], alpha=0.9)
        
        # Marca il punto EER
        eer_idx = np.argmin(np.abs(fpr - metrics['eer']))
        ax.plot(fpr[eer_idx] * 100, fnr[eer_idx] * 100, 'o', 
                markersize=8, color=colors[idx])
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('False Acceptance Rate - FAR (%)')
    ax.set_ylabel('False Rejection Rate - FRR (%)')
    ax.set_title('Detection Error Tradeoff (DET) Curves')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

In [5]:
# ============================================================================
# SEZIONE 4: ANALISI DELLE DISTRIBUZIONI DI DISTANZA
# ============================================================================

def plot_distance_distributions(data, save_path=None):
    """
    Plotta le distribuzioni delle distanze genuine vs impostor
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    for idx, (exp_name, exp_data) in enumerate(data.items()):
        ax = axes[idx]
        metrics = exp_data['metrics'].iloc[0]
        
        # Parse distanze
        genuine_dists = parse_distance_array(metrics['genuine_dists'])
        impostor_dists = parse_distance_array(metrics['impostor_dists'])
        
        # Plot istogrammi
        ax.hist(genuine_dists, bins=50, alpha=0.6, color='green', 
                label='Genuine', density=True, edgecolor='black', linewidth=0.5)
        ax.hist(impostor_dists, bins=50, alpha=0.6, color='red', 
                label='Impostor', density=True, edgecolor='black', linewidth=0.5)
        
        # Aggiungi linee per le medie
        ax.axvline(metrics['mu_genuine'], color='darkgreen', linestyle='--', 
                   linewidth=2, label=f'μ_gen={metrics["mu_genuine"]:.3f}')
        ax.axvline(metrics['mu_impostor'], color='darkred', linestyle='--', 
                   linewidth=2, label=f'μ_imp={metrics["mu_impostor"]:.3f}')
        
        # Soglia EER
        ax.axvline(metrics['eer_threshold'], color='blue', linestyle=':', 
                   linewidth=2, label=f'EER thresh={metrics["eer_threshold"]:.3f}')
        
        ax.set_xlabel('Distance')
        ax.set_ylabel('Density')
        ax.set_title(f'{exp_name}\n(d\'={metrics["d_prime"]:.3f}, Decidability={metrics["decidability"]:.3f})')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()

def analyze_separability(data):
    """
    Analizza la separabilità delle distribuzioni
    """
    separability_stats = []
    
    for exp_name, exp_data in data.items():
        metrics = exp_data['metrics'].iloc[0]
        
        genuine_dists = parse_distance_array(metrics['genuine_dists'])
        impostor_dists = parse_distance_array(metrics['impostor_dists'])
        
        # Statistiche aggiuntive
        overlap_ratio = np.sum((genuine_dists > metrics['eer_threshold'])) / len(genuine_dists)
        
        # Kolmogorov-Smirnov test
        ks_stat, ks_pval = stats.ks_2samp(genuine_dists, impostor_dists)
        
        separability_stats.append({
            'Experiment': exp_name,
            "d' (d-prime)": metrics['d_prime'],
            'Decidability': metrics['decidability'],
            'μ_genuine': metrics['mu_genuine'],
            'μ_impostor': metrics['mu_impostor'],
            'σ_genuine': metrics['sigma_genuine'],
            'σ_impostor': metrics['sigma_impostor'],
            'KS Statistic': ks_stat,
            'KS p-value': ks_pval,
            'Overlap Ratio': overlap_ratio
        })
    
    df_sep = pd.DataFrame(separability_stats)
    return df_sep

In [6]:
# ============================================================================
# SEZIONE 5: ANALISI CROSS-DATASET
# ============================================================================

def compare_intra_vs_cross(data):
    """
    Confronta performance intra-dataset vs cross-dataset
    """
    # Separa esperimenti
    intra_dataset = {k: v for k, v in data.items() if '→' in k and k.split('→')[0] == k.split('→')[1]}
    cross_dataset = {k: v for k, v in data.items() if '→' in k and k.split('→')[0] != k.split('→')[1]}
    
    # Calcola medie
    metrics_to_compare = ['auc', 'eer', 'd_prime', 'decidability', 'accuracy', 'f1']
    
    comparison = []
    for metric in metrics_to_compare:
        intra_values = [exp['metrics'].iloc[0][metric] for exp in intra_dataset.values()]
        cross_values = [exp['metrics'].iloc[0][metric] for exp in cross_dataset.values()]
        
        comparison.append({
            'Metric': metric.upper() if len(metric) <= 3 else metric.replace('_', ' ').title(),
            'Intra-Dataset Mean': np.mean(intra_values),
            'Cross-Dataset Mean': np.mean(cross_values),
            'Absolute Degradation': np.mean(intra_values) - np.mean(cross_values),
            'Relative Degradation (%)': ((np.mean(intra_values) - np.mean(cross_values)) / np.mean(intra_values)) * 100
        })
    
    df_comparison = pd.DataFrame(comparison)
    return df_comparison

def plot_cross_dataset_comparison(data, save_path=None):
    """
    Visualizza confronto tra scenari intra e cross-dataset
    """
    # Prepara dati
    metrics_list = []
    for exp_name, exp_data in data.items():
        metrics = exp_data['metrics'].iloc[0]
        scenario = 'Intra-Dataset' if exp_name.split('→')[0] == exp_name.split('→')[1] else 'Cross-Dataset'
        
        metrics_list.append({
            'Experiment': exp_name,
            'Scenario': scenario,
            'AUC': metrics['auc'],
            'EER': metrics['eer'],
            "d'": metrics['d_prime'],
            'F1': metrics['f1']
        })
    
    df = pd.DataFrame(metrics_list)
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    metrics_to_plot = ['AUC', 'EER', "d'", 'F1']
    
    for idx, metric in enumerate(metrics_to_plot):
        ax = axes[idx // 2, idx % 2]
        
        # Barplot
        x_pos = np.arange(len(df))
        colors_map = {'Intra-Dataset': 'skyblue', 'Cross-Dataset': 'salmon'}
        colors = [colors_map[s] for s in df['Scenario']]
        
        bars = ax.bar(x_pos, df[metric], color=colors, edgecolor='black', linewidth=1.2)
        
        # Aggiungi valori sulle barre
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}',
                    ha='center', va='bottom', fontsize=8)
        
        ax.set_xlabel('Experiment')
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} Comparison')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(df['Experiment'], rotation=45, ha='right')
        ax.grid(True, alpha=0.3, axis='y')
        
        # Aggiungi legenda solo al primo subplot
        if idx == 0:
            from matplotlib.patches import Patch
            legend_elements = [Patch(facecolor='skyblue', edgecolor='black', label='Intra-Dataset'),
                             Patch(facecolor='salmon', edgecolor='black', label='Cross-Dataset')]
            ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()


In [7]:
# ============================================================================
# SEZIONE 6: ANALISI STATISTICA
# ============================================================================

def statistical_significance_tests(data):
    """
    Esegue test statistici per confrontare gli esperimenti
    """
    experiments = list(data.keys())
    results = []
    
    # Test t-test tra le distribuzioni genuine di ogni coppia
    for i in range(len(experiments)):
        for j in range(i+1, len(experiments)):
            exp1_name = experiments[i]
            exp2_name = experiments[j]
            
            metrics1 = data[exp1_name]['metrics'].iloc[0]
            metrics2 = data[exp2_name]['metrics'].iloc[0]
            
            genuine1 = parse_distance_array(metrics1['genuine_dists'])
            genuine2 = parse_distance_array(metrics2['genuine_dists'])
            
            # T-test
            t_stat, t_pval = stats.ttest_ind(genuine1, genuine2)
            
            # Mann-Whitney U test (non-parametric)
            u_stat, u_pval = stats.mannwhitneyu(genuine1, genuine2)
            
            results.append({
                'Comparison': f'{exp1_name} vs {exp2_name}',
                'T-statistic': t_stat,
                'T-test p-value': t_pval,
                'U-statistic': u_stat,
                'Mann-Whitney p-value': u_pval,
                'Significant (α=0.05)': 'Yes' if t_pval < 0.05 else 'No'
            })
    
    df_stats = pd.DataFrame(results)
    return df_stats

In [ ]:
# ============================================================================
# SEZIONE 7: GENERAZIONE REPORT COMPLETO
# ============================================================================

def generate_full_report(base_path, output_dir='./analysis_results'):
    """
    Genera il report completo di analisi
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    print("="*80)
    print("ANALISI COMPLETA ESPERIMENTI DI VERIFICA BIOMETRICA")
    print("Cross-Dataset Writer Verification: IAM ↔ RIMES")
    print("="*80)
    print()
    
    # Carica dati
    print("📂 Caricamento dati...")
    data = load_experiment_data(base_path)
    print("✓ Dati caricati con successo\n")
    
    # 1. Analisi Training
    print("📊 1. ANALISI CURVE DI TRAINING")
    print("-" * 80)
    plot_training_curves(data, save_path=output_path / 'training_curves.png')
    
    conv_stats = analyze_convergence(data)
    print(conv_stats.to_string(index=False))
    conv_stats.to_csv(output_path / 'convergence_statistics.csv', index=False)
    print()
    
    # 2. Performance Summary
    print("📊 2. PERFORMANCE BIOMETRICHE")
    print("-" * 80)
    perf_summary = create_performance_summary(data)
    print(perf_summary.to_string(index=False))
    perf_summary.to_csv(output_path / 'performance_summary.csv', index=False)
    print()
    
    # 3. ROC e DET curves
    print("📊 3. CURVE ROC E DET")
    print("-" * 80)
    plot_roc_curves(data, save_path=output_path / 'roc_curves.png')
    plot_det_curves(data, save_path=output_path / 'det_curves.png')
    print()
    
    # 4. Distribuzioni distanze
    print("📊 4. DISTRIBUZIONI DELLE DISTANZE")
    print("-" * 80)
    plot_distance_distributions(data, save_path=output_path / 'distance_distributions.png')
    
    sep_stats = analyze_separability(data)
    print(sep_stats.to_string(index=False))
    sep_stats.to_csv(output_path / 'separability_analysis.csv', index=False)
    print()
    
    # 5. Confronto Cross-Dataset
    print("📊 5. ANALISI CROSS-DATASET")
    print("-" * 80)
    cross_comparison = compare_intra_vs_cross(data)
    print(cross_comparison.to_string(index=False))
    cross_comparison.to_csv(output_path / 'cross_dataset_comparison.csv', index=False)
    
    plot_cross_dataset_comparison(data, save_path=output_path / 'cross_dataset_barplot.png')
    print()
    
    # 6. Test statistici
    print("📊 6. ANALISI STATISTICA")
    print("-" * 80)
    stat_tests = statistical_significance_tests(data)
    print(stat_tests.to_string(index=False))
    stat_tests.to_csv(output_path / 'statistical_tests.csv', index=False)
    print()
    
    print("="*80)
    print(f"✓ ANALISI COMPLETATA")
    print(f"📁 Tutti i risultati salvati in: {output_path.absolute()}")
    print("="*80)

# ============================================================================
# ESECUZIONE PRINCIPALE
# ============================================================================

if __name__ == "__main__":
    # CONFIGURAZIONE: Modifica questo path con il percorso ai tuoi dati
    BASE_PATH = "../results/bce_experiments"
    OUTPUT_DIR = "../results"
    
    # Genera report completo
    generate_full_report(BASE_PATH, OUTPUT_DIR)
    
    # Per analisi interattive, carica i dati e utilizza le funzioni individualmente:
    # data = load_experiment_data(BASE_PATH)
    # plot_roc_curves(data)
    # perf_summary = create_performance_summary(data)
    # print(perf_summary)

ANALISI COMPLETA ESPERIMENTI DI VERIFICA BIOMETRICA
Cross-Dataset Writer Verification: IAM ↔ RIMES

📂 Caricamento dati...


FileNotFoundError: [Errno 2] No such file or directory: '../results/bce/mobilenet_v3_small_iam_to_iam/mobilenet_v3_small_iam_to_iam_history.csv'